In [1]:
import numpy as np

rng = np.random.default_rng(42)

reference  = rng.normal(loc=0.0, scale=1.0, size=10_000)  # training-time feature
no_drift   = rng.normal(loc=0.0, scale=1.0, size=10_000)  # same distribution
mild_drift = rng.normal(loc=0.3, scale=1.0, size=10_000)  # small mean shift
big_drift  = rng.normal(loc=1.0, scale=1.5, size=10_000)  # shifted mean and wider spread


In [ ]:
# Split the feature into bins, then compare the share of rows in each bin, reference (e) vs current (a):
# PSI = Σ (a − e) × ln(a / e)
# Each bin adds a penalty that grows with how far its share moved. 
# Rule of thumb: below 0.1 stable, 0.1 to 0.25 moderate drift, above 0.25 significant drift.

def psi(reference, current, bins=10, eps=1e-6):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1)) # edges of each bin
    edges[0], edges[-1] = -np.inf, np.inf

    e = np.histogram(reference, bins=edges)[0] / len(reference) # Share of rows in each bin as edges
    a = np.histogram(current, bins=edges)[0] / len(current) # share of rows in each bin of current dist

    e = np.clip(e, eps, None)
    a = np.clip(a, eps, None)
    return np.sum((a - e) * np.log(a / e))

for name, cur in [("no_drift", no_drift), ("mild_drift", mild_drift), ("big_drift", big_drift)]:
    print(name, psi(reference, cur))


no_drift 0.0021992269277573495
mild_drift 0.10101579086277795
big_drift 0.6798786995089124


In [ ]:
# CDF (cumulative distribution function) at a value x = the fraction of data that is ≤ x. 
# It always starts at 0, rises, and ends at 1.
# It jumps up at each data point and stays flat in between. That staircase is called the empirical CDF.


def ks_stat(reference, current):
    ref_sorted = np.sort(reference)
    cur_sorted = np.sort(current)
    points = np.concatenate([ref_sorted, cur_sorted])

    # np.searchsorted([1, 2, 3, 4, 5], 1, side='right') -- no of values below or equal point. 
    cdf_ref = np.searchsorted(ref_sorted, points, side="right") / len(reference)
    cdf_cur = np.searchsorted(cur_sorted, points, side="right") / len(current)
    return np.max(np.abs(cdf_ref - cdf_cur))

for name, cur in [("no_drift", no_drift), ("mild_drift", mild_drift), ("big_drift", big_drift)]:
    print(name, ks_stat(reference, cur))

from scipy.stats import ks_2samp
print(ks_2samp(reference, big_drift))   # statistic should match yours

no_drift 0.016100000000000003
mild_drift 0.13540000000000002
big_drift 0.34659999999999996


KstestResult(statistic=0.3466, pvalue=0.0, statistic_location=0.8697611871649563, statistic_sign=1)
